# Music Graph Demo

**Pipeline:** Synthetic data → Parquet → GCS → FalkorDB → Cypher

This notebook loads a **Music Graph** using direct Parquet upload — no Starburst needed.

Graph shape:
- **Artist** nodes: `artist_id`, `name`, `genre`, `country`
- **Album** nodes: `album_id`, `title`, `year`, `artist_id`
- **Song** nodes: `song_id`, `title`, `duration_secs`, `album_id`
- **RELEASED** edges: Artist → Album
- **CONTAINS** edges: Album → Song

In [ ]:
# ── Step 1 │ Setup & Imports ───────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────
# ── Step 1 │ Setup & Imports ───────────────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────────
import requests, json, time, os
import pandas as pd
from pathlib import Path

API = "http://graph-olap-control-plane:8080"
H   = {"X-Username": "demo@example.com", "X-User-Role": "admin", "Content-Type": "application/json"}

def get(path):        r = requests.get(f"{API}{path}", headers=H);   r.raise_for_status(); return r.json()
def post(path, body): r = requests.post(f"{API}{path}", headers=H, json=body); r.raise_for_status(); return r.json()
def delete(path):     r = requests.delete(f"{API}{path}", headers=H); r.raise_for_status(); return r.json()

print(requests.get(f"{API}/health").json())

## Step 1 — Generate music data

In [ ]:
# ── Step 2 │ Generate Synthetic Data ───────────────────────────────
# Services : local Python — no network
# ──────────────────────────────────────────────────────────────────────
# ── Step 2 │ Generate Synthetic Data ──────────────────────────────────────
# Services : (local Python — no network)
# ──────────────────────────────────────────────────────────────────────────
# --- Artists ---
artists = pd.DataFrame([
    {"artist_id": 1, "name": "The Beatles",   "genre": "Rock",          "country": "UK"},
    {"artist_id": 2, "name": "Led Zeppelin",   "genre": "Hard Rock",     "country": "UK"},
    {"artist_id": 3, "name": "Pink Floyd",     "genre": "Progressive Rock", "country": "UK"},
    {"artist_id": 4, "name": "Radiohead",      "genre": "Alternative",   "country": "UK"},
    {"artist_id": 5, "name": "David Bowie",    "genre": "Glam Rock",     "country": "UK"},
    {"artist_id": 6, "name": "Queen",          "genre": "Rock",          "country": "UK"},
])

# --- Albums ---
albums = pd.DataFrame([
    # The Beatles
    {"album_id": 1,  "title": "Abbey Road",                   "year": 1969, "artist_id": 1},
    {"album_id": 2,  "title": "Sgt. Pepper's Lonely Hearts Club Band", "year": 1967, "artist_id": 1},
    {"album_id": 3,  "title": "Revolver",                     "year": 1966, "artist_id": 1},
    # Led Zeppelin
    {"album_id": 4,  "title": "Led Zeppelin IV",              "year": 1971, "artist_id": 2},
    {"album_id": 5,  "title": "Physical Graffiti",            "year": 1975, "artist_id": 2},
    # Pink Floyd
    {"album_id": 6,  "title": "The Dark Side of the Moon",    "year": 1973, "artist_id": 3},
    {"album_id": 7,  "title": "Wish You Were Here",           "year": 1975, "artist_id": 3},
    # Radiohead
    {"album_id": 8,  "title": "OK Computer",                  "year": 1997, "artist_id": 4},
    {"album_id": 9,  "title": "Kid A",                        "year": 2000, "artist_id": 4},
    # David Bowie
    {"album_id": 10, "title": "Ziggy Stardust",               "year": 1972, "artist_id": 5},
    {"album_id": 11, "title": "Heroes",                       "year": 1977, "artist_id": 5},
    # Queen
    {"album_id": 12, "title": "A Night at the Opera",         "year": 1975, "artist_id": 6},
    {"album_id": 13, "title": "News of the World",            "year": 1977, "artist_id": 6},
])

# --- Songs ---
songs = pd.DataFrame([
    # Abbey Road
    {"song_id": 1,  "title": "Come Together",                 "duration_secs": 259, "album_id": 1},
    {"song_id": 2,  "title": "Something",                     "duration_secs": 182, "album_id": 1},
    {"song_id": 3,  "title": "Here Comes the Sun",            "duration_secs": 185, "album_id": 1},
    # Sgt. Pepper's
    {"song_id": 4,  "title": "Lucy in the Sky with Diamonds", "duration_secs": 208, "album_id": 2},
    {"song_id": 5,  "title": "With a Little Help from My Friends", "duration_secs": 163, "album_id": 2},
    # Revolver
    {"song_id": 6,  "title": "Eleanor Rigby",                 "duration_secs": 127, "album_id": 3},
    {"song_id": 7,  "title": "Yellow Submarine",              "duration_secs": 158, "album_id": 3},
    # Led Zeppelin IV
    {"song_id": 8,  "title": "Stairway to Heaven",            "duration_secs": 482, "album_id": 4},
    {"song_id": 9,  "title": "Black Dog",                     "duration_secs": 296, "album_id": 4},
    {"song_id": 10, "title": "Rock and Roll",                 "duration_secs": 220, "album_id": 4},
    # Physical Graffiti
    {"song_id": 11, "title": "Kashmir",                       "duration_secs": 508, "album_id": 5},
    {"song_id": 12, "title": "Trampled Under Foot",           "duration_secs": 337, "album_id": 5},
    # Dark Side of the Moon
    {"song_id": 13, "title": "Money",                         "duration_secs": 382, "album_id": 6},
    {"song_id": 14, "title": "Time",                          "duration_secs": 421, "album_id": 6},
    {"song_id": 15, "title": "The Great Gig in the Sky",      "duration_secs": 284, "album_id": 6},
    # Wish You Were Here
    {"song_id": 16, "title": "Wish You Were Here",            "duration_secs": 334, "album_id": 7},
    {"song_id": 17, "title": "Shine On You Crazy Diamond",    "duration_secs": 840, "album_id": 7},
    # OK Computer
    {"song_id": 18, "title": "Paranoid Android",              "duration_secs": 383, "album_id": 8},
    {"song_id": 19, "title": "Karma Police",                  "duration_secs": 264, "album_id": 8},
    {"song_id": 20, "title": "No Surprises",                  "duration_secs": 228, "album_id": 8},
    # Kid A
    {"song_id": 21, "title": "Everything in Its Right Place", "duration_secs": 249, "album_id": 9},
    {"song_id": 22, "title": "How to Disappear Completely",   "duration_secs": 355, "album_id": 9},
    # Ziggy Stardust
    {"song_id": 23, "title": "Starman",                       "duration_secs": 254, "album_id": 10},
    {"song_id": 24, "title": "Suffragette City",              "duration_secs": 209, "album_id": 10},
    # Heroes
    {"song_id": 25, "title": "Heroes",                        "duration_secs": 360, "album_id": 11},
    {"song_id": 26, "title": "Beauty and the Beast",          "duration_secs": 221, "album_id": 11},
    # A Night at the Opera
    {"song_id": 27, "title": "Bohemian Rhapsody",             "duration_secs": 354, "album_id": 12},
    {"song_id": 28, "title": "Love of My Life",               "duration_secs": 219, "album_id": 12},
    {"song_id": 29, "title": "You're My Best Friend",         "duration_secs": 170, "album_id": 12},
    # News of the World
    {"song_id": 30, "title": "We Will Rock You",              "duration_secs": 122, "album_id": 13},
    {"song_id": 31, "title": "We Are the Champions",          "duration_secs": 179, "album_id": 13},
])

# --- RELEASED edges: Artist → Album ---
released = albums[["artist_id", "album_id"]].copy()

# --- CONTAINS edges: Album → Song ---
contains = songs[["album_id", "song_id"]].copy()

print(f"Artists: {len(artists)}, Albums: {len(albums)}, Songs: {len(songs)}")
print(f"RELEASED edges: {len(released)}, CONTAINS edges: {len(contains)}")

## Step 2 — Save to Parquet

In [ ]:
# ── Step 3 │ Save to Parquet ───────────────────────────────────────
# Services : local filesystem — no network
# ──────────────────────────────────────────────────────────────────────
# ── Step 3 │ Save to Parquet ───────────────────────────────────────────────
# Services : (local filesystem — no network)
# ──────────────────────────────────────────────────────────────────────────
base = Path("/tmp/music-graph")
for p in ["nodes/Artist", "nodes/Album", "nodes/Song", "edges/RELEASED", "edges/CONTAINS"]:
    (base / p).mkdir(parents=True, exist_ok=True)

artists.to_parquet(base / "nodes/Artist/data.parquet",   index=False)
albums.to_parquet(base  / "nodes/Album/data.parquet",    index=False)
songs.to_parquet(base   / "nodes/Song/data.parquet",     index=False)
released.to_parquet(base / "edges/RELEASED/data.parquet", index=False)
contains.to_parquet(base / "edges/CONTAINS/data.parquet", index=False)

print("Parquet files written to /tmp/music-graph/")

## Step 3 — Create Mapping

In [ ]:
# ── Step 4 │ Create Mapping ────────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────
# ── Step 4 │ Create Mapping ────────────────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────────
mapping = post("/api/mappings", {
    "name": "Music Graph",
    "description": "Artists, albums, songs — real discography dataset",
    "node_definitions": [
        {
            "label": "Artist",
            "sql": "SELECT artist_id, name, genre, country FROM placeholder",
            "primary_key": {"name": "artist_id", "type": "INT64"},
            "properties": [
                {"name": "name",    "type": "STRING"},
                {"name": "genre",   "type": "STRING"},
                {"name": "country", "type": "STRING"},
            ]
        },
        {
            "label": "Album",
            "sql": "SELECT album_id, title, year, artist_id FROM placeholder",
            "primary_key": {"name": "album_id", "type": "INT64"},
            "properties": [
                {"name": "title",     "type": "STRING"},
                {"name": "year",      "type": "INT64"},
                {"name": "artist_id", "type": "INT64"},
            ]
        },
        {
            "label": "Song",
            "sql": "SELECT song_id, title, duration_secs, album_id FROM placeholder",
            "primary_key": {"name": "song_id", "type": "INT64"},
            "properties": [
                {"name": "title",         "type": "STRING"},
                {"name": "duration_secs", "type": "INT64"},
                {"name": "album_id",      "type": "INT64"},
            ]
        },
    ],
    "edge_definitions": [
        {
            "type": "RELEASED",
            "from_node": "Artist",
            "to_node": "Album",
            "sql": "SELECT artist_id, album_id FROM placeholder",
            "from_key": "artist_id",
            "to_key": "album_id",
            "properties": []
        },
        {
            "type": "CONTAINS",
            "from_node": "Album",
            "to_node": "Song",
            "sql": "SELECT album_id, song_id FROM placeholder",
            "from_key": "album_id",
            "to_key": "song_id",
            "properties": []
        },
    ]
})

mapping_id = mapping["data"]["id"]
print(f"Mapping created: id={mapping_id}")

## Step 4 — Create Instance

In [ ]:
# ── Step 5 │ Create Instance + Bypass Export Worker ────────────────
# Services : Control Plane (graph-olap-control-plane:8080)  ·  PostgreSQL (postgres:5432)
# ──────────────────────────────────────────────────────────────────────
# ── Step 5 │ Create Instance + Bypass Export Worker ───────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)  ·  PostgreSQL (postgres:5432)
# ──────────────────────────────────────────────────────────────────────────
inst = post("/api/instances", {
    "mapping_id":    mapping_id,
    "wrapper_type":  "falkordb",
    "name":          "Music Graph Instance",
    "ttl":           "PT4H",
    "description":   "Music graph demo instance",
})
instance_id = inst["data"]["id"]
snapshot_id = inst["data"]["snapshot_id"]
print(f"Instance created: id={instance_id}")
print(f"Snapshot id:      {snapshot_id}")

# Immediately fail any pending export_jobs and mark the snapshot ready.
# This bypasses the export worker (no Starburst needed for local demo).
import psycopg2
conn = psycopg2.connect(
    host="postgres", port=5432,
    dbname="control_plane", user="control_plane", password="control_plane"
)
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute(
        "UPDATE export_jobs SET status='failed' WHERE snapshot_id=%s AND status='pending'",
        (snapshot_id,)
    )
    print(f"  Failed {cur.rowcount} pending export_job(s)")
    cur.execute(
        "UPDATE snapshots SET status='ready' WHERE id=%s",
        (snapshot_id,)
    )
    print(f"  Snapshot {snapshot_id} marked ready")
conn.close()

## Step 5 — Upload Parquet files to GCS

In [ ]:
# ── Step 6 │ Upload Parquet to GCS ─────────────────────────────────
# Services : fake-gcs-local (fake-gcs-local:4443)
# ──────────────────────────────────────────────────────────────────────
# ── Step 6 │ Upload Parquet to GCS ────────────────────────────────────────
# Services : fake-gcs-local (fake-gcs-local:4443)
# ──────────────────────────────────────────────────────────────────────────
# Upload parquet files directly to fake-gcs via HTTP API
BUCKET = "graph-olap-local-dev"
GCS    = "http://fake-gcs-local:4443"
OWNER  = "demo@example.com"
PREFIX = f"{OWNER}/{mapping_id}/v1/{snapshot_id}"
print(f"Uploading to gs://{BUCKET}/{PREFIX}/")

# Ensure bucket exists in fake-gcs (in-memory — recreated after each pod restart)
_br = requests.post(f"{GCS}/storage/v1/b", json={"name": BUCKET},
               headers={"Content-Type": "application/json"})
if _br.status_code not in (200, 201, 409):
    _br.raise_for_status()

files = [
    ("nodes/Artist/data.parquet",   f"{PREFIX}/nodes/Artist/data.parquet"),
    ("nodes/Album/data.parquet",    f"{PREFIX}/nodes/Album/data.parquet"),
    ("nodes/Song/data.parquet",     f"{PREFIX}/nodes/Song/data.parquet"),
    ("edges/RELEASED/data.parquet", f"{PREFIX}/edges/RELEASED/data.parquet"),
    ("edges/CONTAINS/data.parquet", f"{PREFIX}/edges/CONTAINS/data.parquet"),
]

for local, remote in files:
    file_path = base / local
    with open(file_path, "rb") as f:
        data = f.read()
    url  = f"{GCS}/upload/storage/v1/b/{BUCKET}/o?uploadType=media&name={remote}"
    resp = requests.post(url, data=data, headers={"Content-Type": "application/octet-stream"})
    if resp.status_code in (200, 201):
        print(f"  OK  {remote}")
    else:
        print(f"  ERR {remote}: {resp.status_code} {resp.text[:200]}")

## Step 6 — Poll for instance running status

In [ ]:
# ── Step 7 │ Wait for Instance ─────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────
# ── Step 7 │ Wait for Instance ────────────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────────
print("Polling... (reconciliation loop runs every ~30s)")
start = time.time()
while True:
    elapsed = int(time.time() - start)
    inst    = get(f"/api/instances/{instance_id}")
    status  = inst["data"]["status"]
    print(f"  [{elapsed:3d}s] instance={status}")
    if status == "running":
        break
    if status in ("stopped", "failed"):
        raise RuntimeError(f"Instance ended with status: {status}")
    time.sleep(10)

print("\nMusic graph is running! FalkorDB pod ready.")

## Step 7 — Query the Music Graph

In [ ]:
# ── Step 8 │ Connect & Query Graph ─────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# ── Step 8 │ Connect & Query Graph ────────────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────────
inst_data = get(f"/api/instances/{instance_id}")["data"]
pod_name  = inst_data["pod_name"]
WRAPPER   = f"http://{pod_name}:8000"
print(f"Wrapper: {WRAPPER}")

def cypher(q, params=None):
    body = {"query": q}
    if params:
        body["parameters"] = params
    for attempt in range(10):
        try:
            r = requests.post(f"{WRAPPER}/query",
                              headers={"Content-Type": "application/json"}, json=body)
            r.raise_for_status()
            return r.json()["rows"]
        except Exception as e:
            if attempt < 9:
                print(f"  Wrapper not ready yet, retrying in 3s... ({attempt+1}/10)")
                time.sleep(3)
            else:
                raise

# --- Node counts ---
print("=== Graph loaded — Node counts ===")
for row in cypher("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS cnt ORDER BY cnt DESC"):
    print(f"  {str(row[0]):20s}: {row[1]:,}")

In [ ]:
# ── Step 8a │ Cypher — Albums per Artist ───────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# ── Step 8a │ Cypher — Albums per Artist ──────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────────
# --- Albums by artist ---
print("=== Albums per artist ===")
results = cypher("""
    MATCH (ar:Artist)-[:RELEASED]->(al:Album)
    WITH ar.name AS artist, ar.genre AS genre, count(al) AS album_count,
         collect(al.title) AS albums
    RETURN artist, genre, album_count, albums
    ORDER BY album_count DESC
""")
for row in results:
    print(f"  {row[0]:20s} ({row[1]:20s})  {row[2]} album(s): {row[3]}")

In [ ]:
# ── Step 8b │ Cypher — Longest Songs ───────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# ── Step 8b │ Cypher — Longest Songs ──────────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────────
# --- Longest songs ---
print("=== Top 10 longest songs ===")
results = cypher("""
    MATCH (ar:Artist)-[:RELEASED]->(al:Album)-[:CONTAINS]->(s:Song)
    RETURN s.title AS song, ar.name AS artist, al.title AS album,
           s.duration_secs AS duration_secs
    ORDER BY duration_secs DESC
    LIMIT 10
""")
for row in results:
    mins, secs = divmod(row[3], 60)
    print(f"  {row[0]:40s}  {row[1]:20s}  {mins}:{secs:02d}")

In [ ]:
# ── Step 8c │ Cypher — Artists by Album Count ──────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# ── Step 8c │ Cypher — Artists by Album Count ──────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────────
# --- Artists with most albums ---
print("=== Artists ranked by number of albums ===")
results = cypher("""
    MATCH (ar:Artist)-[:RELEASED]->(al:Album)
    WITH ar.name AS artist, ar.genre AS genre, count(al) AS albums
    RETURN artist, genre, albums
    ORDER BY albums DESC
""")
for row in results:
    print(f"  {row[0]:20s}  {row[1]:20s}  {row[2]} album(s)")

In [ ]:
# ── Step 8d │ Cypher — Songs per Genre ─────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# ── Step 8d │ Cypher — Songs per Genre ────────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────────
# --- Songs per genre (via Artist genre) ---
print("=== Song count per genre ===")
results = cypher("""
    MATCH (ar:Artist)-[:RELEASED]->(al:Album)-[:CONTAINS]->(s:Song)
    WITH ar.genre AS genre, count(s) AS song_count
    RETURN genre, song_count
    ORDER BY song_count DESC
""")
for row in results:
    print(f"  {str(row[0]):25s}: {row[1]} song(s)")

## Step 9 — Visualise (PyVis)

In [ ]:
# ── Step 9 │ Visualise Graph (PyVis) ───────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)  ·  PyVis (local)
# ──────────────────────────────────────────────────────────────────────
# ── Step 9 │ Visualise Graph (PyVis) ──────────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)  ·  PyVis (local)
# ──────────────────────────────────────────────────────────────────────────
from pyvis.network import Network
from IPython.display import IFrame

net = Network(height="600px", width="100%", bgcolor="#1a1a2e", font_color="white", notebook=True)

added_nodes = set()

rows = cypher("""
    MATCH (ar:Artist)-[:RELEASED]->(al:Album)-[:CONTAINS]->(s:Song)
    RETURN ar.artist_id, ar.name, al.album_id, al.title, s.song_id, s.title
""")

for row in rows:
    ar_id, ar_name, al_id, al_title, s_id, s_title = row

    ar_key = f"artist_{ar_id}"
    if ar_key not in added_nodes:
        net.add_node(ar_key, label=ar_name, color="#e94560", shape="star", size=30)
        added_nodes.add(ar_key)

    al_key = f"album_{al_id}"
    if al_key not in added_nodes:
        net.add_node(al_key, label=al_title, color="#0f3460", shape="dot", size=22)
        added_nodes.add(al_key)

    s_key = f"song_{s_id}"
    if s_key not in added_nodes:
        net.add_node(s_key, label=s_title, color="#533483", shape="square", size=12)
        added_nodes.add(s_key)

    net.add_edge(ar_key, al_key, color="orange", title="RELEASED")
    net.add_edge(al_key, s_key,  color="grey",   title="CONTAINS")

net.save_graph("music_graph.html")
IFrame("music_graph.html", width="100%", height=620)

## Cleanup (optional)

In [ ]:
# ── Step 10 │ Cleanup ──────────────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────
# ── Step 10 │ Cleanup ─────────────────────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────────
# Clean up — delete the instance to free the wrapper pod memory
try:
    delete(f"/api/instances/{instance_id}")
    print(f"Instance {instance_id} deleted — wrapper pod will be removed")
except Exception as e:
    print(f"Instance {instance_id} already gone or not found: {e}")